# Build the DiffusionFace test set

Crops a fixed sample of DiffusionFace part 1 (Chen et al., arXiv 2403.18471) with the same RetinaFace pipeline as `build_dataset.ipynb`, and uploads it as a **private** Hugging Face dataset, the test side of the `cross_diffusionface` protocol. The lab never trains on it. Unlike DiFF, the real photos are open, so this set gives an AUC on diffusion faces right away.

**Access:** part 1 is open on [Zenodo](https://zenodo.org/records/10865300) under CC BY 4.0, and the authors ask for research use only. Each folder is one tar of about 3 GB, 50 GB in all. Zenodo served each connection at about 0.6 MB/s from Colab in October 2026, and parallel connections added up, so the notebook downloads all tars at once (about 1.5 hours) before cropping the sampled image numbers.

**Sample:** the authors publish no test split. `diffusionface.sampled` keeps about one image number in `EVERY`, the same numbers in every folder, so each real photo sits next to the fakes made from it.

**Runtime:** Colab Pro with a GPU and 55 GB of free disk. **Needs** the Colab secret `HF_TOKEN` with write access, for the upload cell.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-diffusionface-test"  # private
BUILD_DIR = "/content/build_diffusionface"
TAR_DIR = "/content/diffusionface_tars"
ZENODO_RECORD = "https://zenodo.org/records/10865300/files"
# MM_CelebA.tar holds only the captions behind the text prompts, no images.
FOLDERS = ("Real", "ADM", "DDIM", "DDPM", "PNDM", "LDM", "SDv15_DS0.3", "SDv15_DS0.5",
           "SDv15_DS0.7", "SDv21_DS0.3", "SDv21_DS0.5", "SDv21_DS0.7", "DiffSwap", "Inpaint")
EVERY = 20  # about 1,500 image numbers per folder

### Installation

Same as `build_dataset.ipynb`: `onnxruntime-gpu` and the ONNX backend, so RetinaFace runs on the GPU.

In [ ]:
%%capture
!pip install -q "face-decision-lab[build] @ {REPO_SOURCE}" gdown
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu

In [ ]:
import os
os.environ["DEEPFACE_BACKEND_ENGINE"] = "onnx"

import onnxruntime as ort
providers = ort.get_available_providers()
print("ONNX Runtime providers:", providers)
assert "CUDAExecutionProvider" in providers, "onnxruntime-gpu cannot see the GPU; RetinaFace would run on the CPU"

### Download

All tars in parallel. `curl -C -` resumes a partial file, so rerunning the cell after a dropped connection continues where it stopped; finished tars are marked `.done` and skipped.

In [ ]:
import os, subprocess
from pathlib import Path

os.makedirs(TAR_DIR, exist_ok = True)
pending = [f for f in FOLDERS if not Path(f"{TAR_DIR}/{f}.done").exists()]
downloads = [subprocess.Popen(["curl", "-sSL", "--retry", "10", "-C", "-", "-o", f"{TAR_DIR}/{folder}.tar",
                               f"{ZENODO_RECORD}/{folder}.tar?download=1"]) for folder in pending]
failed = []
for folder, process in zip(pending, downloads):
    if process.wait() == 0:
        Path(f"{TAR_DIR}/{folder}.done").touch()
    else:
        failed.append(folder)
assert not failed, f"downloads failed for {failed}; rerun this cell to resume them"
for folder in FOLDERS:
    print(f"{folder}: {os.path.getsize(f'{TAR_DIR}/{folder}.tar') / 1e9:.2f} GB")

### Build

Resumable, like the main build: a rerun skips every image already in `processed.jsonl`. Each tar is deleted after cropping.

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
from face_decision_lab.build import DatasetBuilder, tar_members
from face_decision_lab.faces import RetinaFaceDetector
from face_decision_lab.sources import diffusionface

builder = DatasetBuilder(BUILD_DIR, RetinaFaceDetector())
for folder in tqdm(FOLDERS, desc = "tars"):
    path = Path(TAR_DIR) / f"{folder}.tar"
    if not path.exists():  # cropped and deleted by an earlier run
        continue
    with path.open("rb") as stream:
        added = builder.add_images(tar_members(stream, diffusionface.parse_member,
                                               keep = lambda name: diffusionface.sampled(name, EVERY)))
    builder.flush()
    path.unlink()
    print(f"{folder}: {added} images")
print(len(builder.samples), "crops,", sum(s.is_bona_fide for s in builder.samples), "bona fide")

### Check detection outcomes

A detector that drops one folder more often than the others biases the test; compare the saved share per generator.

In [ ]:
for generator in sorted(builder.report.counts):
    counts = builder.report.counts[generator]
    print(f"{generator:28s} saved {builder.report.saved_share(generator):6.1%}  {counts}")

### Upload as a private dataset

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from face_decision_lab.hub import push_build

login(token = userdata.get("HF_TOKEN"))
print(push_build(BUILD_DIR, HF_DATASET))